In [6]:
import numpy as np

class ART1:
    def __init__(self, n, m, rho=0.5, alpha=1.5):
        self.n = n
        self.m = m
        self.rho = rho
        self.alpha = alpha
        # bij: i × j (rows = input features, cols = nodes)
        self.bij = np.ones((n, m)) / (alpha + n - 1)
        # tji: j × i (rows = nodes, cols = input features)
        self.tji = np.ones((m, n))

    def _norm(self, x):
        return np.sum(np.abs(x))

    def _check_binary(self, x):
        if not np.isin(x, [0, 1]).all():
            raise ValueError("ART1 only accepts binary inputs.")

    def _initialize_layers(self, x):
        return x.copy()

    def _compute_activations(self, s):
        # yj = Σ bij[i,j] * xi
        y = np.zeros(self.m)
        for j in range(self.m):
            y[j] = np.dot(self.bij[:, j], s)
        return y

    def _select_node_and_check_reset(self, s, y):
        while True:
            J = np.argmax(y)
            if y[J] == -1:
                print("All nodes inhibited — stopping search.")
                return None,None
                
            x_hat = s * self.tji[J]  # element-wise
            ratio = self._norm(x_hat) / (self._norm(s))
            
            print(f"Node {J}: ||x||/||s|| = {ratio:.4f}, vigilance = {self.rho}")
            
            if ratio < self.rho:
                print(f"→ Reset TRUE → Inhibit node {J}")
                y[J] = -1
                continue
            else:
                print(f"→ Resonance achieved with node {J}")
                return J, x_hat

    def _print_weights(self, J):
        print(f"\nUpdated bij (i×j) after node {J} update:")
        
        for i in range(self.n):
            print(" ".join(f"{self.bij[i, j]:.1f}" for j in range(self.m)))

        print(f"\nUpdated tji (j×i) after node {J} update:")
        
        for j in range(self.m):
            print(" ".join(f"{self.tji[j, i]:.1f}" for i in range(self.n)))
        print()

    def _update_weights(self, J, x_hat):
        
        numerator = self.alpha * x_hat
        denominator = (self.alpha - 1) + self._norm(x_hat)
        
        # Update bij (i × j)
        self.bij[:, J] = numerator / denominator
        
        # Update tji (j × i)
        self.tji[J, :] = x_hat.copy()
        self._print_weights(J)

    def train(self, X, epochs=100):
        X = np.array(X)
        self._check_binary(X)
        print("=== ART1 Training Started ===")
        for epoch in range(epochs):
            print(f"\n=== Epoch {epoch+1} ===")
            weight_changed = False
            for idx, x in enumerate(X):
                print(f"\nInput {idx+1}: {x}")
                s = self._initialize_layers(x)
                y = self._compute_activations(s)
                print(f"Activations y: {y}")
                J, x_hat = self._select_node_and_check_reset(s, y)
                if J is None:
                    continue
                print(f"Updating weights for node {J}")
                self._update_weights(J, x_hat)
                weight_changed = True
            if not weight_changed:
                print("Stopping condition met — No weight change in this epoch.")
                break
        print("\n=== Training Completed ===\n")

    def predict(self, X):
        X = np.array(X)
        self._check_binary(X)
        labels = []
        for x in X:
            y = np.dot(self.bij.T, x)  # bij: i×j → need transpose
            labels.append(np.argmax(y))
        return np.array(labels)


if __name__ == "__main__":
    x = np.array([
        [0, 0, 0, 1],
        [0, 1, 0, 1],
        [0, 0, 1, 1],
        [1, 0, 0, 0]
    ])

    art = ART1(n=4, m=3, rho=0.4, alpha=2)
    art.train(x, epochs=5)
    print("Final Cluster Assignments:")
    print(art.predict(x))

=== ART1 Training Started ===

=== Epoch 1 ===

Input 1: [0 0 0 1]
Activations y: [0.2 0.2 0.2]
Node 0: ||x||/||s|| = 1.0000, vigilance = 0.4
→ Resonance achieved with node 0
Updating weights for node 0

Updated bij (i×j) after node 0 update:
0.0 0.2 0.2
0.0 0.2 0.2
0.0 0.2 0.2
1.0 0.2 0.2

Updated tji (j×i) after node 0 update:
0.0 0.0 0.0 1.0
1.0 1.0 1.0 1.0
1.0 1.0 1.0 1.0


Input 2: [0 1 0 1]
Activations y: [1.  0.4 0.4]
Node 0: ||x||/||s|| = 0.5000, vigilance = 0.4
→ Resonance achieved with node 0
Updating weights for node 0

Updated bij (i×j) after node 0 update:
0.0 0.2 0.2
0.0 0.2 0.2
0.0 0.2 0.2
1.0 0.2 0.2

Updated tji (j×i) after node 0 update:
0.0 0.0 0.0 1.0
1.0 1.0 1.0 1.0
1.0 1.0 1.0 1.0


Input 3: [0 0 1 1]
Activations y: [1.  0.4 0.4]
Node 0: ||x||/||s|| = 0.5000, vigilance = 0.4
→ Resonance achieved with node 0
Updating weights for node 0

Updated bij (i×j) after node 0 update:
0.0 0.2 0.2
0.0 0.2 0.2
0.0 0.2 0.2
1.0 0.2 0.2

Updated tji (j×i) after node 0 update:
0.0